# Local neurotransmitter mapping

In this tutorial, you will explore the local neurotransmitter mapping (LNTM) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch and prepare the neurotransmitter PET atlas
- Compute per-target neurotransmitter density scores within a lesion
- Filter by neurotransmitter system presets
- Obtain parcel-level NT scores

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/local-neurotransmitter-mapping.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

Get the tutorial data.

In [ ]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force

## Fetch neurotransmitter atlas

Local neurotransmitter mapping requires a neurotransmitter PET atlas: a collection of PET receptor/transporter density maps from normative cohorts. Lacuna downloads these from the [Open Science Framework](https://osf.io/yz9mb/).

The atlas includes maps for multiple neurotransmitter targets (e.g., D1, 5HT1a, DAT, GABAa) derived from published PET tracer studies.

In [ ]:
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

## Prepare the atlas

Before running the analysis, the raw PET maps need to be prepared. This step:
1. Groups maps by neurotransmitter target
2. Averages maps within each target (excluding zeros)
3. Z-scores the result
4. Caches the processed atlas for reuse

In [ ]:
!lacuna prepare lntm \
    --source-dir /tmp/ntatlas_data

## Analysis

Local neurotransmitter mapping scores the z-scored PET atlas values directly within the lesion mask. For each neurotransmitter target, it computes the mean (or sum) of the atlas values at lesion voxels.

This answers: **what neurotransmitter landscape did the lesion wipe out?**

A high score for a given target indicates that the lesioned region is rich in that neurotransmitter system, suggesting potential neurochemical consequences of the lesion.

Run the analysis.

In [ ]:
!lacuna run lntm \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntm/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym

List the outputs.

In [ ]:
!ls /tmp/outputs_lntm/sub-01/ses-01/anat/

Visualize the per-target neurotransmitter scores.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load the LNTM scores from the JSON sidecar
results_dir = Path("/tmp/outputs_lntm/sub-01/ses-01/anat/")
score_files = sorted(results_dir.glob("*method-lntm*scores*.json"))

# If scores are in TSV format instead
import pandas as pd
tsv_files = sorted(results_dir.glob("*method-lntm*parcelstats.tsv"))

if tsv_files:
    df = pd.read_csv(tsv_files[0], sep="\t")
    print(df)

## Filter by neurotransmitter system

Instead of computing all targets, you can restrict the analysis to a specific neurotransmitter system using the `--targets` flag with a preset name:

| Preset | Targets |
|--------|--------|
| `dopaminergic` | D1, D23, DAT, FDOPA |
| `serotonergic` | 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT |
| `cholinergic` | VAChT, M1, A4B2 |
| `monoaminergic` | D1, D23, DAT, 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT, NET |
| `all` | All available targets (default) |

You can also pass a comma-separated list of individual targets (e.g., `--targets D1,DAT,5HT2a`).

In [ ]:
!lacuna run lntm \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntm_dopamine/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --targets dopaminergic

## Obtain parcel-level NT scores

Beyond global lesion-level scores, Lacuna can also compute NT scores per atlas parcel. This provides a spatial profile of the neurotransmitter landscape across brain regions.

In [ ]:
!lacuna run lntm \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntm_parcels/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --parcel-atlases schaefer2018parcels100networks7 \
    --verbose

In [ ]:
!ls /tmp/outputs_lntm_parcels/sub-01/ses-01/anat/

## Run on multiple subjects

Lacuna supports processing multiple subjects within a single run. If the `--participant-label` flag is omitted, the pipeline automatically processes all subjects detected in the BIDS dataset.

LNTM is very fast since it only requires voxel lookups in the atlas — no connectome loading is needed.

In [ ]:
!lacuna run lntm \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntm_all/ \
    --mask-space MNI152NLin6Asym

Collect results into a group-level table.

In [ ]:
!lacuna collect \
    /tmp/outputs_lntm_all/ \
    --pattern "*lntm*parcelstats*" \
    --output-dir /tmp/group_lntm/

In [ ]:
!ls /tmp/group_lntm/